# Shared memory

Section 14.2 of the notes.

When the workers read one large array that none of them changes, the array can
be placed in a block of memory that every process reads directly, instead of
being sent to each worker.

In [ ]:
import os
import time
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import shared_memory

import numpy as np

import tasks

N = os.cpu_count()


def wall(fn):
    t0 = time.perf_counter()
    fn()
    return time.perf_counter() - t0

## 1. Putting an array in shared memory

In [ ]:
rng = np.random.default_rng(0)
data = rng.standard_normal(20_000_000)  # 160 MB

shm = shared_memory.SharedMemory(create=True, size=data.nbytes)
shared = np.ndarray(data.shape, dtype=data.dtype, buffer=shm.buf)
shared[:] = data  # copied once

print("name  ", shm.name)
print("shape ", shared.shape)
print("MB    ", shared.nbytes / 1e6)

Each task receives only the name of the block and the range it should read, so
the array itself is never sent.

In [ ]:
edges = np.linspace(0, shared.size, N + 1, dtype=int)
args = [
    (shm.name, shared.shape, str(shared.dtype), int(edges[i]), int(edges[i + 1]))
    for i in range(N)
]

with ProcessPoolExecutor(max_workers=N) as pool:
    t_shared = wall(lambda: sum(pool.map(tasks.chunk_work, args)))

pieces = [data[int(edges[i]) : int(edges[i + 1])] for i in range(N)]
with ProcessPoolExecutor(max_workers=N) as pool:
    t_sent = wall(lambda: sum(pool.map(tasks.sum_of_copy, pieces)))

t_here = wall(lambda: tasks.harmonics(data))

print(f"shared memory      {t_shared:5.2f} s")
print(f"sending the pieces {t_sent:5.2f} s")
print(f"one process        {t_here:5.2f} s")

The result has to agree with the direct computation, and the block has to be
released. `close` detaches this process, `unlink` removes the block.

In [ ]:
with ProcessPoolExecutor(max_workers=N) as pool:
    total_shared = sum(pool.map(tasks.chunk_work, args))

print("agrees:", np.isclose(total_shared, tasks.harmonics(data)))

shm.close()
shm.unlink()
print("block released")

## 2. When it is worth it

In the measurement above the two parallel versions cost the same. The array is
split once, so each worker receives its piece exactly once, and that single
transfer is cheap compared with the work on it. Both are about three times
faster than the single process, and the gain comes from the work per entry.

Shared memory pays off in a different situation: when many short tasks read the
same large array, so that sending it would be repeated for every task. It never
removes the memory traffic itself, so a computation that only reads each entry
once and computes almost nothing from it stays limited by the memory, however
many workers read it.

## 3. One thread per worker

The routines behind matrix products start threads of their own. A pool of 18
processes, each running such a routine on 18 threads, creates far more threads
than the machine has cores, and they slow each other down. The usual fix is to
give every worker one thread, either with the environment variables below,
before NumPy is imported, or with `threadpool_limits` inside the worker.

On this machine NumPy uses Apple's Accelerate library, which manages its own
threads and reports none to `threadpoolctl`, so the call below has no effect
here. On a machine with OpenBLAS or MKL, which is the usual case on Linux, it
does.

In [ ]:
import os

# set before numpy is imported, in every worker
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "1")

from threadpoolctl import threadpool_info, threadpool_limits  # noqa: E402

print("pools numpy reports here:", threadpool_info())
with threadpool_limits(limits=1):
    a = np.random.default_rng(0).standard_normal((1500, 1500))
    t = wall(lambda: a @ a)
print(f"a matrix product with one thread requested: {t:.3f} s")